# Setup

In [1]:
import torch

# Seed
seed=42

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Summary

```markdown
Tokenizer
├── Normalizer
├── PreTokenizer      ← Whitespace / ByteLevel / Metaspace / BertPreTokenizer
├── Model             ← BPE / WordPiece / Unigram
├── PostProcessor
└── Decoder
```

# 1. API

In [ ]:
from transformers import AutoTokenizer

# Load
model_name = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer_bck = tokenizer.backend_tokenizer

# Structure
print(f"[Normalizer] {tokenizer_bck.normalizer}")
print(f"[Pre-tokenizer] {tokenizer_bck.pre_tokenizer}")
print(f"[Model] {tokenizer_bck.model}")
print(f"[Post-processor] {tokenizer_bck.post_processor}", '\n')

# Text
text = [
    "Hello, transformer!",
    "What is your name?",
]
print(f"[Text] {text}")

# Encode: text -> id
encode = tokenizer(
    text,
    padding="max_length",
    truncation=True,
    max_length=10,
    return_tensors="pt",
)
print(f"[Encode] {encode['input_ids']}")

# Decode: id -> text
print(f"[Decode] {tokenizer.decode(encode['input_ids'], skip_special_tokens=False)}")
print(f"[Decode] {tokenizer.batch_decode(encode['input_ids'], skip_special_tokens=True)}")

# token id -> token
tokens = tokenizer.convert_ids_to_tokens(encode['input_ids'][0])
print(f"[id -> token] {tokens}")

# token -> id
ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"[token -> id] {ids}")

# Methods and attributes
tokenizer.is_fast               # whether loaded as a fast tokenizer.

tokenizer.all_special_tokens    # ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']
tokenizer.all_special_ids       # [100, 102, 0, 101, 103]
tokenizer.pad_token
tokenizer.unk_token
tokenizer.bos_token
tokenizer.eos_token

tokenizer.padding_side          # left / right.
tokenizer.truncation_side       # left / right.

tokenizer.get_vocab()           # get vocab.
tokenizer.vocab_size            # vocab size.

tokenizer.model_max_length      # maximum n_tokens of the transformer.

[Normalizer] BertNormalizer(clean_text=True, handle_chinese_chars=True, strip_accents=None, lowercase=True)
[Pre-tokenizer] BertPreTokenizer()
[Model] WordPiece(unk_token="[UNK]", continuing_subword_prefix="##", max_input_chars_per_word=100, vocab={"[PAD]":0, "[unused0]":1, "[unused1]":2, "[unused2]":3, "[unused3]":4, ...})
[Post-processor] TemplateProcessing(single=[SpecialToken(id="[CLS]", type_id=0), Sequence(id=A, type_id=0), SpecialToken(id="[SEP]", type_id=0)], pair=[SpecialToken(id="[CLS]", type_id=0), Sequence(id=A, type_id=0), SpecialToken(id="[SEP]", type_id=0), Sequence(id=B, type_id=1), SpecialToken(id="[SEP]", type_id=1)], special_tokens={"[CLS]":SpecialToken(id="[CLS]", ids=[101], tokens=["[CLS]"]), "[SEP]":SpecialToken(id="[SEP]", ids=[102], tokens=["[SEP]"])}) 

[Text] ['Hello, transformer!', 'What is your name?']
[Encode] tensor([[  101,  7592,  1010, 10938,  2121,   999,   102,     0,     0,     0],
        [  101,  2054,  2003,  2115,  2171,  1029,   102,     0,     

512

# 2. Normalizer

| API | Description | Before | After |
|---|---|---|---|
| `Lowercase()` | Converts text to lowercase. | `AI` | `ai` |
| `Strip()` | Removes leading/trailing whitespace. | `\n he\t\tllo \n` | `he\t\tllo` |
| `StripAccents()` | Removes combining accent marks. | `cafe\u0301` | `cafe` |
| `NFD()` | Canonically decomposes Unicode characters. | `café` | `cafe\u0301` |
| `NFC()` | Canonically composes Unicode characters. | `cafe\u0301` | `café` |
| `NFKD()` | Decomposes Unicode including compatibility characters. | `ﬃ` | `ffi` |
| `NFKC()` | Composes Unicode including compatibility normalization. | `Ｈｅｌｌｏ` | `Hello` |
| `Replace('world', 'Python')` | Replaces matching text. | `Hello world` | `Hello Python` |
| `ByteLevel()` | Maps UTF-8 bytes to visible Unicode characters. | `hello\nworld` | `helloĊworld` |
| `Prepend('▁')` | Adds a prefix to the text. | `hello` | `▁hello` |
| `BertNormalizer()` | Applies BERT-style cleaning, lowercasing, and accent removal. | `Héllo WORLD` | `hello world` |
| `Nmt()` | Applies SentencePiece NMT-style text cleaning. | `Hello\x00World` | `Hello World` |
| `Precompiled(charsmap_bytes)` | Applies a SentencePiece model's normalization map. | `Ｈｅｌｌｏ` | Model-dependent |
| `Sequence([NFD(), StripAccents()])` | Applies multiple normalizers sequentially. | `café` | `cafe` |

In [ ]:
from tokenizers import normalizers

# From backend tokenizer
normalizer = tokenizer.backend_tokenizer.normalizer

# Lowercase
text = 'AI' 
norm_lower = normalizers.Lowercase()
text_norm = norm_lower.normalize_str(text)
print(f"Lowercase(): {text} -> {text_norm}")

# Strip
text = '\n he\t\tllo \n '
norm_strip = normalizers.Strip()
text_norm = norm_strip.normalize_str(text)
print(f"Strip(): {text!r} -> {text_norm!r}")

# Replace
text = 'Hello'
norm_replace = normalizers.Replace('H', 'h')
text_norm = norm_replace.normalize_str(text)
print(f"Replace(): {text} -> {text_norm}")

# NFD (Normalization Form D)
text = 'café'
text_uni = [ord(c) for c in text]
norm_nfd = normalizers.NFD()
text_norm = norm_nfd.normalize_str(text)
text_norm_uni = [ord(c) for c in text_norm]
print(f"NFD(): {text_uni} -> {text_norm_uni}")

# Sequence / StripAccents
text = 'café'
norm_accent = normalizers.Sequence(
    [
        normalizers.NFD(),
        normalizers.StripAccents(),   
    ]
)
text_norm = norm_accent.normalize_str(text)
print(f"NFD() + StripAccents(): {text} -> {text_norm}")

# ByteLevel
text = 'hello, world!\n'
norm_byte = normalizers.ByteLevel()
text_norm = norm_byte.normalize_str(text)
print(f"ByteLevel(): {text!r} -> {text_norm!r}")

# Prepend
text = 'hello, world!'
norm_prepend = normalizers.Prepend('▁')
text_norm = norm_prepend.normalize_str(text)
print(f"Prepend(): {text} -> {text_norm}")

Lowercase(): AI -> ai
Strip(): '\n he\t\tllo \n ' -> 'he\t\tllo'
Replace(): Hello -> hello
NFD(): [99, 97, 102, 233] -> [99, 97, 102, 101, 769]
NFD() + StripAccents(): café -> cafe
ByteLevel(): 'hello, world!\n' -> 'hello,Ġworld!Ċ'
Prepend(): hello, world! -> ▁hello, world!


# 3. Pre-tokenizer

| API | Description | Before | After |
|---|---|---|---|
| `BertPreTokenizer()` | Splits on whitespace and individual punctuation marks. | `Hello, world!` | `['Hello', ',', 'world', '!']` |
| `ByteLevel(add_prefix_space=False)` | Splits using GPT-2 rules and maps bytes to Unicode. | `Hello world!` | `['Hello', 'Ġworld', '!']` |
| `CharDelimiterSplit(',')` | Splits on a specified character, removing it. | `apple,banana,cherry` | `['apple', 'banana', 'cherry']` |
| `Digits(individual_digits=True)` | Separates individual digits from text. | `abc123def` | `['abc', '1', '2', '3', 'def']` |
| `FixedLength(length=3)` | Splits text into fixed-length chunks. | `abcdefgh` | `['abc', 'def', 'gh']` |
| `Metaspace()` | Replaces spaces with `▁` and splits at boundaries. | `Hello world!` | `['▁Hello', '▁world!']` |
| `Punctuation()` | Separates individual punctuation characters. | `Hello...world!` | `['Hello', '.', '.', '.', 'world', '!']` |
| `Split(',', behavior='removed')` | Splits using a custom pattern. | `apple,banana,cherry` | `['apple', 'banana', 'cherry']` |
| `UnicodeScripts()` | Splits at Unicode script boundaries. | `HelloПривет世界` | `['Hello', 'Привет', '世界']` |
| `Whitespace()` | Splits on whitespace and separates punctuation groups. | `Hello, world!` | `['Hello', ',', 'world', '!']` |
| `WhitespaceSplit()` | Splits only on whitespace. | `Hello, world!` | `['Hello,', 'world!']` |
| `Sequence([...])` | Applies multiple pre-tokenizers sequentially. | `Call 911!` | `['Call', '9', '1', '1', '!']` |

In [ ]:
from tokenizers import pre_tokenizers

# BertPreTokenizer
text = "Hello, world!"
pre_bert = pre_tokenizers.BertPreTokenizer()
text_pre = pre_bert.pre_tokenize_str(text)
print(f"BertPreTokenizer(): {text!r} -> {text_pre}")

# ByteLevel
text = "Hello world!"
pre_byte = pre_tokenizers.ByteLevel(add_prefix_space=False)
text_pre = pre_byte.pre_tokenize_str(text)
print(f"ByteLevel(): {text!r} -> {text_pre}")

# CharDelimiterSplit
text = "apple,banana,cherry"
pre_char = pre_tokenizers.CharDelimiterSplit(",")
text_pre = pre_char.pre_tokenize_str(text)
print(f"CharDelimiterSplit(): {text!r} -> {text_pre}")

# Digits
text = "abc123def"
pre_digits = pre_tokenizers.Digits(individual_digits=True)
text_pre = pre_digits.pre_tokenize_str(text)
print(f"Digits(): {text!r} -> {text_pre}")

# FixedLength
text = "abcdefgh"
pre_fixed = pre_tokenizers.FixedLength(length=3)
text_pre = pre_fixed.pre_tokenize_str(text)
print(f"FixedLength(): {text!r} -> {text_pre}")

# Metaspace: whitespace -> _
text = "Hello w orld!"
pre_meta = pre_tokenizers.Metaspace(replacement="▁")
text_pre = pre_meta.pre_tokenize_str(text)
print(f"Metaspace(): {text!r} -> {text_pre}")

# Punctuation
text = "Hello...world!"
pre_punct = pre_tokenizers.Punctuation()
text_pre = pre_punct.pre_tokenize_str(text)
print(f"Punctuation(): {text!r} -> {text_pre}")

# Sequence
text = "Call 911!"
pre_seq = pre_tokenizers.Sequence([
    pre_tokenizers.Whitespace(),
    pre_tokenizers.Digits(individual_digits=True)
])
text_pre = pre_seq.pre_tokenize_str(text)
print(f"Sequence(): {text!r} -> {text_pre}")

# Split: delimiter + behavior
text = "apple,banana,cherry"
pre_split = pre_tokenizers.Split(pattern=",", behavior="removed")   # behavior = {'removed', 'isolated', 'merged_with_previous', 'merged_with_next', 'contiguous'}
text_pre = pre_split.pre_tokenize_str(text)
print(f"Split(): {text!r} -> {text_pre}")

# UnicodeScripts
text = "HelloПривет世界"
pre_unicode = pre_tokenizers.UnicodeScripts()
text_pre = pre_unicode.pre_tokenize_str(text)
print(f"UnicodeScripts(): {text!r} -> {text_pre}")

# Whitespace: whitespace + punctuation
text = "Hello, world!"
pre_ws = pre_tokenizers.Whitespace()
text_pre = pre_ws.pre_tokenize_str(text)
print(f"Whitespace(): {text!r} -> {text_pre}")

# WhitespaceSplit: whitespace
text = "Hello, world!"
pre_ws_split = pre_tokenizers.WhitespaceSplit()
text_pre = pre_ws_split.pre_tokenize_str(text)
print(f"WhitespaceSplit(): {text!r} -> {text_pre}")

BertPreTokenizer(): 'Hello, world!' -> [('Hello', (0, 5)), (',', (5, 6)), ('world', (7, 12)), ('!', (12, 13))]
ByteLevel(): 'Hello world!' -> [('Hello', (0, 5)), ('Ġworld', (5, 11)), ('!', (11, 12))]
CharDelimiterSplit(): 'apple,banana,cherry' -> [('apple', (0, 5)), ('banana', (6, 12)), ('cherry', (13, 19))]
Digits(): 'abc123def' -> [('abc', (0, 3)), ('1', (3, 4)), ('2', (4, 5)), ('3', (5, 6)), ('def', (6, 9))]
FixedLength(): 'abcdefgh' -> [('abc', (0, 3)), ('def', (3, 6)), ('gh', (6, 8))]
Metaspace(): 'Hello w orld!' -> [('▁Hello', (0, 5)), ('▁w', (5, 7)), ('▁orld!', (7, 13))]
Punctuation(): 'Hello...world!' -> [('Hello', (0, 5)), ('.', (5, 6)), ('.', (6, 7)), ('.', (7, 8)), ('world', (8, 13)), ('!', (13, 14))]
Sequence(): 'Call 911!' -> [('Call', (0, 4)), ('9', (5, 6)), ('1', (6, 7)), ('1', (7, 8)), ('!', (8, 9))]
Split(): 'apple,banana,cherry' -> [('apple', (0, 5)), ('banana', (6, 12)), ('cherry', (13, 19))]
UnicodeScripts(): 'HelloПривет世界' -> [('Hello', (0, 5)), ('Привет', (5, 11)

# 4. Model

- Word-level Tokenizer
  - "I love you" -> "I", "love", "you"
  - Out-of-vocabulary -> not used in modern LLMs.
- Character-level Tokenizer
  - "love" -> "l", "o", "v", "e"
  - Large sequence length -> not used in modern LLMs.
- BPE, Byte-pair Encoding
  - Character-level BPE
    - starts with each character -> counts all pairs -> merges the frequent pairs until it reaches the vocab size.
  - Byte-level BPE
    - starts with each byte in unicode to cover unicode characters.
  - SentencePiece BPE
    - uses `pre_tokenizers.Metaspace()`, i.e. whitespace -> '_' and encodes within 'the_whole_sentences'.
  - BPE-dropout
    - intentionally skips some merges.
  - Tokenization
    - Merge-rank-based tokenization: the highest-priority available pair.
- WordPiece
  - $Score(A, B) \propto \frac{freq(A, B)}{freq(A)freq(B)}$
  - Tokenization
    - Greedy longest-match-first tokenization: the longest matching prefix.
- Unigram
  - Starts with large candidates -> measures the explainability -> prunes the weak candidates.
  - Subword regularization: randomly choose the candidate from top K instead of the best one.

In [17]:
corpus = [
    "I love machine learning",
    "I love natural language processing",
    "machine learning is interesting",
    "natural language processing is interesting",
    "learning machines can learn",
]

## 4.1. Character-Level BPE

In [34]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers

tokenizer = Tokenizer(models.BPE(unk_token="<UNK>"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()   # character-level.

# Train
trainer = trainers.BpeTrainer(
    vocab_size=300,
    min_frequency=1,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),   # start with unicode characters.
    special_tokens=["<UNK>"],
)
tokenizer.train_from_iterator(corpus, trainer)

# Tokenize
text = "I love machine killing!"
print(tokenizer.encode(text).tokens)

['I', 'love', 'machine', 'k', 'i', 'l', 'l', 'ing', '!']


## 4.2. Byte-Level BPE

In [36]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders

tokenizer = Tokenizer(models.BPE(unk_token="<UNK>"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()    # byte-level.
tokenizer.decoder = decoders.ByteLevel()

# Train
trainer = trainers.BpeTrainer(
    vocab_size=300,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),   # start with unicode characters.
    special_tokens=["<UNK>"],
)
tokenizer.train_from_iterator(corpus, trainer)

# Tokenize
text = "I love machine killing!"
print(tokenizer.encode(text).tokens)
print(tokenizer.decode(tokenizer.encode(text).ids))

['ĠI', 'Ġlov', 'e', 'Ġmachine', 'Ġ', 'k', 'i', 'l', 'l', 'ing', '!']
 I love machine killing!


## 4.3. SentencePiece BPE

In [37]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders

tokenizer = Tokenizer(models.BPE(unk_token="<UNK>"))
tokenizer.pre_tokenizer = pre_tokenizers.Metaspace(replacement="▁",)
tokenizer.decoder = decoders.Metaspace(replacement="▁",)

# Train
trainer = trainers.BpeTrainer(
    vocab_size=50,
    min_frequency=1,
    special_tokens=["<UNK>"],
)

# Tokenize
tokenizer.train_from_iterator(corpus, trainer)
print(tokenizer.encode(text).tokens)

['▁', 'I', '▁l', 'ov', 'e', '▁machin', 'e', '▁', '<UNK>', 'i', 'l', 'l', 'ing', '<UNK>']


## 4.4. WordPiece

In [ ]:
tokenizer = Tokenizer(
    models.WordPiece(unk_token="[UNK]")
)
tokenizer.pre_tokenizer = pre_tokenizers.BertPreTokenizer()

# Train
trainer = trainers.WordPieceTrainer(
    vocab_size=50,
    min_frequency=1,
    special_tokens=["[UNK]"],
)
tokenizer.train_from_iterator(corpus, trainer)

# Tokenize
print(tokenizer.encode(text).tokens)

['I', 'lo', '##v', '##e', 'machin', '##e', '[UNK]', '[UNK]']


## 4.5. Unigram

In [ ]:
tokenizer = Tokenizer(models.Unigram())
tokenizer.pre_tokenizer = pre_tokenizers.Metaspace(replacement="▁")
tokenizer.decoder = decoders.Metaspace(replacement="▁")

# Train
trainer = trainers.UnigramTrainer(
    vocab_size=50,
    special_tokens=["<UNK>"],
    unk_token="<UNK>",
)

# Tokenize
tokenizer.train_from_iterator(corpus, trainer)
print(tokenizer.encode(text).tokens)

['▁', 'I', '▁l', 'o', 'v', 'e', '▁machine', '▁', 'k', 'i', 'l', 'l', 'in', 'g', '!']


# 5. Post-processor

| API | Description | Before | After |
|---|---|---|---|
| `BertProcessing()` | Adds BERT special tokens `[CLS]` and `[SEP]`. | `['Hello', 'world']` | `['[CLS]', 'Hello', 'world', '[SEP]']` |
| `RobertaProcessing()` | Adds RoBERTa special tokens `<s>` and `</s>` and optionally trims offsets. | `['Hello', 'world']` | `['<s>', 'Hello', 'world', '</s>']` |
| `ByteLevel()` | Trims whitespace from token offsets without changing tokens. | `('ĠHello', (0, 6))` | `('ĠHello', (1, 6))` |
| `TemplateProcessing()` | Adds special tokens using customizable templates. | `['Hello', 'world']` | `['<bos>', 'Hello', 'world', '<eos>']` |
| `Sequence()` | Applies multiple post-processors sequentially. | `['ĠHello']` | `['<bos>', 'ĠHello', '<eos>']` |

In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, processors

# Simple tokenizer for demonstration
vocab = {
    "[UNK]": 0,
    "Hello": 1,
    "world": 2,
    "[CLS]": 3,
    "[SEP]": 4,
    "<s>": 5,
    "</s>": 6,
    "<bos>": 7,
    "<eos>": 8,
    "ĠHello": 9,
}

tokenizer = Tokenizer(models.WordLevel(vocab=vocab, unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.WhitespaceSplit()

# BertProcessing
text = "Hello world"
encoding = tokenizer.encode(text, add_special_tokens=False)

post_bert = processors.BertProcessing(("[SEP]", 4), ("[CLS]", 3))
encoding_post = post_bert.process(encoding)
print(f"BertProcessing(): {encoding.tokens} -> {encoding_post.tokens}")


# RobertaProcessing
text = "Hello world"
encoding = tokenizer.encode(text, add_special_tokens=False)

post_roberta = processors.RobertaProcessing(("</s>", 6), ("<s>", 5))
encoding_post = post_roberta.process(encoding)
print(f"RobertaProcessing(): {encoding.tokens} -> {encoding_post.tokens}")


# ByteLevel
text = " Hello"

byte_tokenizer = Tokenizer(models.WordLevel(vocab=vocab, unk_token="[UNK]"))
byte_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
byte_encoding = byte_tokenizer.encode(text, add_special_tokens=False)

post_byte = processors.ByteLevel(add_prefix_space=False, trim_offsets=True)
encoding_post = post_byte.process(byte_encoding)
print(f"ByteLevel(): {byte_encoding.offsets} -> {encoding_post.offsets}")


# TemplateProcessing
text = "Hello world"
encoding = tokenizer.encode(text, add_special_tokens=False)

post_template = processors.TemplateProcessing(
    single="<bos> $A <eos>",
    pair="<bos> $A <eos> $B:1 <eos>:1",
    special_tokens=[("<bos>", 7), ("<eos>", 8)]
)
encoding_post = post_template.process(encoding)
print(f"TemplateProcessing(): {encoding.tokens} -> {encoding_post.tokens}")


# Sequence
text = " Hello"
encoding = byte_tokenizer.encode(text, add_special_tokens=False)

post_seq = processors.Sequence([
    processors.ByteLevel(add_prefix_space=False, trim_offsets=True),
    processors.TemplateProcessing(
        single="<bos> $A <eos>",
        special_tokens=[("<bos>", 7), ("<eos>", 8)]
    )
])
encoding_post = post_seq.process(encoding)
print(f"Sequence(): {encoding.tokens} -> {encoding_post.tokens}")

BertProcessing(): ['Hello', 'world'] -> ['[CLS]', 'Hello', 'world', '[SEP]']
RobertaProcessing(): ['Hello', 'world'] -> ['<s>', 'Hello', 'world', '</s>']
ByteLevel(): [(0, 6)] -> [(1, 6)]
TemplateProcessing(): ['Hello', 'world'] -> ['<bos>', 'Hello', 'world', '<eos>']
Sequence(): ['ĠHello'] -> ['<bos>', 'ĠHello', '<eos>']
PostProcessor: True
